# E-Commerce Price Predictor: Full Stack Generator 🚀
Run the cells below to install dependencies, train your model, test the model with metrics & graphs, and automatically generate the server and frontend code files.

In [ ]:
!pip install pandas numpy scikit-learn joblib fastapi uvicorn pydantic matplotlib seaborn

## 1. Data Preparation and Train/Test Split
First, we load the data and split it into separate training and testing sets so we can properly evaluate the model's performance on unseen data.

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

print("Loading dataset...")
try:
    df = pd.read_csv('train.tsv', sep='\t', nrows=50000) # Subset for quick training
    df['category_name'] = df['category_name'].fillna('Missing')
    df['brand_name'] = df['brand_name'].fillna('Missing')
    df['item_description'] = df['item_description'].fillna('No description yet')

    df = df[df['price'] > 0].reset_index(drop=True)
    df['combined_text'] = df['name'] + " " + df['item_description']
    
    X = df[['combined_text', 'item_condition_id', 'shipping']]
    y = np.log1p(df['price']) # Predict log of price for better regression performance

    print("Splitting into Training and Testing data...")
    # 80% Training data, 20% Testing data
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    
    print(f"Training samples: {len(X_train)}")
    print(f"Testing samples: {len(X_test)}")
except FileNotFoundError:
    print("Ensure train.tsv is in the same directory.")

Loading dataset...
Splitting into Training and Testing data...
Training samples: 39968
Testing samples: 9992


## 2. Train and Save the Machine Learning Model
We train the model exclusively on the training set (`X_train` and `y_train`).

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
import joblib

preprocessor = ColumnTransformer(
    transformers=[
        ('text', TfidfVectorizer(max_features=10000, stop_words='english'), 'combined_text')
    ],
    remainder='passthrough'
)

model_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', Ridge(alpha=1.0, solver='lsqr'))
])

print("Training model on the Training Set...")
model_pipeline.fit(X_train, y_train)

joblib.dump(model_pipeline, 'mercari_price_model.joblib')
print("Model successfully saved to mercari_price_model.joblib")

Training model on the Training Set...


TypeError: cg() got an unexpected keyword argument 'tol'

## 3. Test and Evaluate Data (Metrics and Graphs)
Because predicting price is a **Regression** task (predicting a continuous number), we usually use Regression metrics (MAE, RMSE, R²). However, since you requested Classification metrics (Accuracy, Precision, Recall, F1 Score, and a Confusion Matrix), we will evaluate the regression model AND also categorize the predicted prices into "Budget", "Mid-Range", and "Premium" to calculate those classification metrics and draw the confusion matrix!

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, ConfusionMatrixDisplay

print("Evaluating the model on the unseen Test Data...")
# Predict on test data
y_pred_log = model_pipeline.predict(X_test)
y_pred = np.expm1(y_pred_log)  # Convert back from log scale
y_actual = np.expm1(y_test)

# --- REGRESSION METRICS ---
mae = mean_absolute_error(y_actual, y_pred)
rmse = np.sqrt(mean_squared_error(y_actual, y_pred))
r2 = r2_score(y_actual, y_pred)

print("\n=== REGRESSION METRICS ===")
print(f"Mean Absolute Error (MAE): ${mae:.2f}")
print(f"Root Mean Squared Error (RMSE): ${rmse:.2f}")
print(f"R-squared (R²): {r2:.4f}")

# Regression Plots
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.scatter(y_actual, y_pred, alpha=0.3, color='blue')
plt.plot([0, max(y_actual)], [0, max(y_actual)], color='red', linestyle='--')
plt.title('Regression: Actual vs Predicted Price')
plt.xlabel('Actual Price ($)')
plt.ylabel('Predicted Price ($)')
plt.xlim(0, 300)
plt.ylim(0, 300)

plt.subplot(1, 2, 2)
residuals = y_actual - y_pred
sns.histplot(residuals, bins=50, kde=True, color='purple')
plt.title('Regression: Residuals Distribution')
plt.xlabel('Error (Actual - Predicted) ($)')
plt.xlim(-100, 100)

plt.tight_layout()
plt.show()

# --- CLASSIFICATION METRICS & CONFUSION MATRIX ---
print("\n=== CLASSIFICATION METRICS (Categorized Prices) ===")
def categorize_price(price):
    if price <= 20: return 'Budget (<=$20)'
    elif price <= 50: return 'Mid-Range ($20-$50)'
    else: return 'Premium (>$50)'

y_actual_cat = [categorize_price(p) for p in y_actual]
y_pred_cat = [categorize_price(p) for p in y_pred]

labels = ['Budget (<=$20)', 'Mid-Range ($20-$50)', 'Premium (>$50)']

acc = accuracy_score(y_actual_cat, y_pred_cat)
prec = precision_score(y_actual_cat, y_pred_cat, average='weighted', zero_division=0)
rec = recall_score(y_actual_cat, y_pred_cat, average='weighted', zero_division=0)
f1 = f1_score(y_actual_cat, y_pred_cat, average='weighted', zero_division=0)

print(f"Accuracy: {acc*100:.2f}%")
print(f"Weighted Precision: {prec:.4f}")
print(f"Weighted Recall: {rec:.4f}")
print(f"Weighted F1-Score: {f1:.4f}\n")

cm = confusion_matrix(y_actual_cat, y_pred_cat, labels=labels)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels)
fig, ax = plt.subplots(figsize=(6,6))
disp.plot(cmap='Blues', ax=ax, xticks_rotation=45)
plt.title('Confusion Matrix (Price Categories)')
plt.tight_layout()
plt.show()

## 4. Generate the FastAPI Backend (`server.py`)

In [ ]:
%%writefile server.py
from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
import joblib
import numpy as np
import pandas as pd

app = FastAPI(title="E-Commerce Price Predictor API")

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

model = joblib.load('mercari_price_model.joblib')

class ProductPredictionInput(BaseModel):
    name: str
    item_description: str
    item_condition_id: int
    shipping: int

@app.post("/predict")
def predict_price(data: ProductPredictionInput):
    combined_text = f"{data.name} {data.item_description}"
    input_df = pd.DataFrame([{
        'combined_text': combined_text,
        'item_condition_id': data.item_condition_id,
        'shipping': data.shipping
    }])
    
    log_prediction = model.predict(input_df)[0]
    final_price = float(np.expm1(log_prediction))
    return {"predicted_price": round(max(0.0, final_price), 2)}

if __name__ == "__main__":
    import uvicorn
    uvicorn.run(app, host="0.0.0.0", port=8000)

## 5. Generate the React Frontend (`App.jsx`)

In [ ]:
%%writefile App.jsx
import React, { useState } from 'react';

export default function App() {
  const [formData, setFormData] = useState({
    name: '',
    item_description: '',
    item_condition_id: 1,
    shipping: 0
  });
  const [prediction, setPrediction] = useState(null);
  const [loading, setLoading] = useState(false);

  const handleSubmit = async (e) => {
    e.preventDefault();
    setLoading(true);
    try {
      const response = await fetch('http://localhost:8000/predict', {
        method: 'POST',
        headers: { 'Content-Type': 'application/json' },
        body: JSON.stringify({
          ...formData,
          item_condition_id: parseInt(formData.item_condition_id),
          shipping: parseInt(formData.shipping)
        })
      });
      const data = await response.json();
      setPrediction(data.predicted_price);
    } catch (error) {
      console.error("Error fetching prediction:", error);
    } finally {
      setLoading(false);
    }
  };

  return (
    <div className="min-h-screen bg-gray-50 flex flex-col items-center justify-center p-6">
      <div className="max-w-xl w-full bg-white shadow-md rounded-lg p-8 border border-gray-200">
        <h1 className="text-2xl font-bold text-gray-900 mb-6 text-center">
          E-Commerce Price Predictor
        </h1>
        
        <form onSubmit={handleSubmit} className="space-y-5">
          <div>
            <label className="block text-sm font-semibold text-gray-700 mb-1">Product Title</label>
            <input
              type="text"
              required
              placeholder="e.g., Nike Air Max 90"
              className="w-full px-4 py-2 border border-gray-300 rounded-md"
              value={formData.name}
              onChange={(e) => setFormData({...formData, name: e.target.value})}
            />
          </div>
          <div>
            <label className="block text-sm font-semibold text-gray-700 mb-1">Item Description</label>
            <textarea
              required
              rows="4"
              placeholder="Provide a detailed item description..."
              className="w-full px-4 py-2 border border-gray-300 rounded-md"
              value={formData.item_description}
              onChange={(e) => setFormData({...formData, item_description: e.target.value})}
            />
          </div>
          <button
            type="submit"
            disabled={loading}
            className="w-full bg-blue-600 hover:bg-blue-700 text-white font-medium py-2.5 px-4 rounded-md"
          >
            {loading ? 'Evaluating Parameters...' : 'Calculate Target Price'}
          </button>
        </form>

        {prediction !== null && (
          <div className="mt-8 pt-6 border-t border-gray-200 text-center">
            <p className="text-sm uppercase tracking-wider font-semibold text-gray-500">Predicted Optimal Value</p>
            <p className="text-4xl font-extrabold text-green-600 mt-1">${prediction}</p>
          </div>
        )}
      </div>
    </div>
  );
}
